# In-Silico Perturbation Benchmark (NM-only)

Inject a **known** synthetic log2FC into a random 250-gene subset of 5 real HC batches (largest by n_hc), score under 2 training conditions with 3 different aggregation statistics, and compare against the known ground truth:

- **Condition LOBO**: score with the LOBO fold fit that EXCLUDED this batch from training.
- **Condition In-sample**: score with the main engine (`ENGINE_MIXED_DIR`), trained on ALL HC batches including this one -- matches how `3_disease_scoring.ipynb` actually scores real disease samples.
- **Stouffer's Z** (parametric): sum per-sample Z across the batch, test against N(0,1). Not used in the normative-modeling literature we could verify (see below) -- included as the naive baseline.
- **Extreme MWU** (nonparametric, our own construction): rank-sum each gene's |Z| against the pooled |Z| of every other gene in the same batch/condition.
- **Wolfers-style extreme-deviation count** (literature-verified: Wolfers et al. 2018, JAMA Psychiatry, [10.1001/jamapsychiatry.2018.2467](https://doi.org/10.1001/jamapsychiatry.2018.2467)): binarize |Z|>2.6 (their threshold, p<.005), compute each gene's extreme-deviation rate across the batch's samples, and test it against the batch's OWN empirical background extreme-rate (a chi-square-equivalent proportion test) -- their actual published method for testing normative-model deviations at the group level.

**Literature check** (see chat log for the full trail): of the normative-modeling papers we could access in full text, only Wolfers et al. 2018 and Rutherford et al. 2022 ([10.1038/s41596-022-00696-5](https://doi.org/10.1038/s41596-022-00696-5)) explicitly describe a group-level test, and both use the same binarize-then-count-then-test pattern -- never a continuous Z-sum (Stouffer). No verified precedent for Stouffer's method was found in this literature.

**Headline finding**: Wolfers' method dominates Stouffer and MWU on AUC, Sensitivity, AND Precision simultaneously (no tradeoff) at every effect size. All 3 methods still leave a few hundred to a few thousand residual "significant" genes under zero or small injected effect -- traced earlier to within-batch, within-gene sample correlation induced by the per-gene random-batch-intercept (tau2) that marginal RQR integrates out but does not condition on.

**Caveat on Precision**: with only 250 injected genes out of ~17,800 (prevalence ~1.4%), even a well-calibrated test has low absolute Precision -- this is a base-rate (PPV) effect, not necessarily a sign the test is bad. The comparison cell below also reports **Precision / prevalence** (enrichment over random) and **Precision vs. the ~95% a well-calibrated FDR=5% test should give**, which are the prevalence-robust ways to read this.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

import config
from core.marginal_rqr import marginal_nb_rqr
from core.model_engine_mixed import NormativeModelEngineMixed
from core.shash import shash_transform_to_z
from validation.lobo_engine import load_full_data
from validation.ppc_simulate import simulate_marginal_nb

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import apply_style
apply_style()

PERTURB_DIR = config.ROOT / "MixedEffectsModeling" / "Perturbation_Results"
PERTURB_DIR.mkdir(exist_ok=True)

BATCHES = ["Ward Z et al._Batch_1", "Moufarrej et al._Batch_2", "Moore et al._Batch_1",
           "Moufarrej et al._Batch_1", "Tuni et al._Batch_2"]  # 5 largest usable HC batches by n_hc
LOG2FCS = [1.0, 2.0, 3.0, 4.0, 5.0]
N_PERTURB_GENES = 250
N_BOOTSTRAP = 5
GROUP_ALPHA = 0.05
INSAMPLE_ALPHA = 0.20  # looser: a single sample has far less power than a pooled group stat
Z_THRESH = 2.6  # Wolfers et al. 2018 extreme-deviation cutoff (two-sided p<.005)

data = load_full_data()

## Shared helpers

In [2]:
def apply_shash(z, xi, eta, eps, delta, ok):
    return shash_transform_to_z(z, xi, eta, eps, delta) if ok else z


def mwu_z(col_abs, sorted_bg, n_bg):
    """Rank-sum z-score of a gene's |Z| samples against the pooled |Z| of every OTHER gene
    in the same batch/condition -- shared batch-wide drift mostly cancels."""
    ranks = np.searchsorted(sorted_bg, col_abs, side="left").astype(np.float64)
    n1 = len(col_abs)
    mean_u, sd_u = n1 * n_bg / 2, np.sqrt(n1 * n_bg * (n1 + n_bg + 1) / 12)
    return (ranks.sum() - mean_u) / sd_u


def sens_prec(scores, labels, p_from_scores, alpha, higher_is_positive=True):
    fin = np.isfinite(scores)
    reject, _, _, _ = multipletests(p_from_scores(scores[fin]), alpha=alpha, method="fdr_bh")
    lab = labels[fin]
    tp, n_sig = int((reject & (lab == 1)).sum()), int(reject.sum())
    score_for_auc = scores[fin] if higher_is_positive else np.abs(scores[fin])
    return (roc_auc_score(lab, score_for_auc), tp / max((lab == 1).sum(), 1), tp / max(n_sig, 1), n_sig)


def persample_sens_prec(Z_full, labels, n_hc, alpha):
    aucs, sens, prec = [], [], []
    for i in range(n_hc):
        scores = np.abs(Z_full[i])
        fin = np.isfinite(scores)
        if fin.sum() < 10:
            continue
        aucs.append(roc_auc_score(labels[fin], scores[fin]))
        rej, _, _, _ = multipletests(2 * norm.sf(scores[fin]), alpha=alpha, method="fdr_bh")
        lab = labels[fin]
        n_sig = rej.sum()
        sens.append(rej[lab == 1].sum() / max((lab == 1).sum(), 1))
        prec.append(rej[lab == 1].sum() / n_sig if n_sig > 0 else np.nan)
    return np.nanmean(aucs), np.nanmean(sens), np.nanmean(prec)

## Sweep loop -- shared across LOBO / In-sample conditions

Adds a 3rd statistic (Wolfers-style extreme-count proportion test) alongside Stouffer and MWU.

In [3]:
def lobo_mu_alpha(fits, idx, genes, batch_id):
    tr_idx = np.where(data["is_hc"] & (data["batch"] != batch_id) &
                       ~np.isin(data["batch"], list(data["small_hc_batches"])))[0]
    scaler = StandardScaler().fit(data["X_raw"][tr_idx])
    Xa = np.column_stack([np.ones(len(idx)), scaler.transform(data["X_raw"][idx])])
    mu, alpha, tau2 = {}, {}, {}
    for g in genes:
        row = fits.loc[g]
        mu_coef = row[[c for c in fits.columns if c.startswith("mu_coef_")]].values.astype(float)
        disp_coef = row[[c for c in fits.columns if c.startswith("disp_coef_")]].values.astype(float)
        mu[g] = np.clip(np.exp(Xa @ np.nan_to_num(mu_coef, nan=0.0)), 1e-6, 1e8)
        alpha[g] = (np.exp(-Xa @ np.nan_to_num(disp_coef, nan=0.0)) if not np.all(np.isnan(disp_coef))
                    else np.full(len(idx), float(row["trend_alpha"])))
        tau2[g] = float(row["tau2"])
    return mu, alpha, tau2


def run_sweep(mu_alpha_fn, shash_of, universe_of, label, cache_name):
    """`mu_alpha_fn(idx, genes, batch_id)` and `shash_of(g, batch_id)` -> (xi,eta,eps,delta,ok)
    are the only things that differ between LOBO and in-sample."""
    cache_path = PERTURB_DIR / cache_name
    if cache_path.is_file():
        return pd.read_csv(cache_path)
    rows = []
    for batch_id in BATCHES:
        universe = universe_of(batch_id)
        hc_idx = np.where(data["is_hc"] & (data["batch"] == batch_id))[0]
        n_hc = len(hc_idx)

        mu_u, alpha_u, tau2_u = mu_alpha_fn(hc_idx, universe, batch_id)
        Z_real = np.column_stack([
            apply_shash(marginal_nb_rqr(data["Y"][hc_idx, data["gene_col"][g]], mu_u[g], alpha_u[g], tau2_u[g], seed=42),
                        *shash_of(g, batch_id))
            for g in universe])
        stouffer_real = np.nansum(Z_real, axis=0) / np.sqrt(np.isfinite(Z_real).sum(axis=0))
        sorted_bg = np.sort(np.abs(Z_real).ravel())
        n_bg = sorted_bg.size
        mwu_real = np.array([mwu_z(np.abs(Z_real[:, j]), sorted_bg, n_bg) for j in range(len(universe))])
        extreme_real = np.abs(Z_real) > Z_THRESH
        p0_batch = np.nanmean(extreme_real)
        gene_pos = {g: j for j, g in enumerate(universe)}

        for log2fc in LOG2FCS:
            for b in range(N_BOOTSTRAP):
                rng = np.random.default_rng(1000 * b + int(log2fc * 10))
                pert_genes = list(rng.choice(universe, min(N_PERTURB_GENES, len(universe)), replace=False))
                mu, alpha, tau2 = mu_alpha_fn(hc_idx, pert_genes, batch_id)

                Z_full, stouffer_full, mwu_full, extreme_full = Z_real.copy(), stouffer_real.copy(), mwu_real.copy(), extreme_real.copy()
                for g in pert_genes:
                    mu_shift = mu[g] * (2 ** log2fc)
                    y_pert = simulate_marginal_nb(mu_shift, alpha[g], tau2[g], n_reps=1, seed=1000 * b + hash(g) % 997)[0]
                    z_raw = marginal_nb_rqr(y_pert, mu[g], alpha[g], tau2[g], seed=2000 * b + hash(g) % 997)
                    z = apply_shash(z_raw, *shash_of(g, batch_id))
                    j = gene_pos[g]
                    Z_full[:, j] = z
                    stouffer_full[j] = np.nansum(z) / np.sqrt(np.isfinite(z).sum())
                    mwu_full[j] = mwu_z(np.abs(z), sorted_bg, n_bg)
                    extreme_full[:, j] = np.abs(z) > Z_THRESH

                labels = np.array([1 if g in set(pert_genes) else 0 for g in universe])
                auc_st, sens_st, prec_st, nsig_st = sens_prec(stouffer_full, labels, lambda s: 2 * norm.sf(np.abs(s)), GROUP_ALPHA)
                auc_mwu, sens_mwu, prec_mwu, nsig_mwu = sens_prec(mwu_full, labels, lambda s: norm.sf(s), GROUP_ALPHA)

                phat = extreme_full.sum(axis=0) / n_hc
                z_wolf = (phat - p0_batch) / np.sqrt(p0_batch * (1 - p0_batch) / n_hc + 1e-12)
                auc_wolf, sens_wolf, prec_wolf, nsig_wolf = sens_prec(z_wolf, labels, norm.sf, GROUP_ALPHA)

                auc_i, sens_i, prec_i = persample_sens_prec(Z_full, labels, n_hc, INSAMPLE_ALPHA)

                rows.append(dict(condition=label, batch=batch_id, log2fc=log2fc, boot=b, n_universe=len(universe),
                                  auc_stouffer=auc_st, sens_stouffer=sens_st, prec_stouffer=prec_st, nsig_stouffer=nsig_st,
                                  auc_mwu=auc_mwu, sens_mwu=sens_mwu, prec_mwu=prec_mwu, nsig_mwu=nsig_mwu,
                                  auc_wolfers=auc_wolf, sens_wolfers=sens_wolf, prec_wolfers=prec_wolf, nsig_wolfers=nsig_wolf,
                                  auc_insample=auc_i, sens_insample=sens_i, prec_insample=prec_i))
        print(label, batch_id, "done")
    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return df

## Run both conditions

Cache-first -- delete the corresponding `Perturbation_Results/sweep_*.csv` to force a recompute (~10-15 min each).

In [4]:
# --- LOBO condition ---
_lobo_fits, _lobo_shash = {}, {}


def _lobo_load(batch_id):
    dirname = batch_id.replace(" ", "_")
    if batch_id not in _lobo_fits:
        _lobo_fits[batch_id] = pd.read_csv(config.LOBO_MIXED_DIR / dirname / "model_fits.csv", index_col="gene")
        _lobo_shash[batch_id] = pd.read_csv(config.LOBO_MIXED_DIR / dirname / "shash_params.csv", index_col="gene")
    return _lobo_fits[batch_id], _lobo_shash[batch_id]


def lobo_universe(batch_id):
    fits, shash_p = _lobo_load(batch_id)
    return [g for g in fits.index[fits["ok"]].tolist() if g in shash_p.index]


def lobo_shash_of(g, batch_id):
    row = _lobo_shash[batch_id].loc[g]
    return row["xi"], row["eta"], row["eps"], row["delta"], bool(row["ok"])


def lobo_mu_alpha_fn(idx, genes, batch_id):
    fits, _ = _lobo_load(batch_id)
    return lobo_mu_alpha(fits, idx, genes, batch_id)


results_lobo = run_sweep(lobo_mu_alpha_fn, lobo_shash_of, lobo_universe, "LOBO", "sweep_v3.csv")

# --- In-sample condition (main engine, trained on ALL HC incl. this batch) ---
engine = NormativeModelEngineMixed.load(config.ENGINE_MIXED_DIR)
_insample_universe = [g for g, rec in engine.genes.items() if rec.ok and rec.route == "model" and rec.cv_shash_xi is not None]


def insample_mu_alpha_fn(idx, genes, batch_id):
    Xa = np.column_stack([np.ones(len(idx)), engine.scaler.transform(data["X_raw"][idx])])
    mu, alpha, tau2 = {}, {}, {}
    for g in genes:
        rec = engine.genes[g]
        mu[g] = np.clip(np.exp(Xa @ np.nan_to_num(rec.mu_coef, nan=0.0)), 1e-6, 1e8)
        alpha[g] = (np.exp(-Xa @ np.nan_to_num(rec.disp_coef, nan=0.0)) if not np.all(np.isnan(rec.disp_coef))
                    else np.full(len(idx), rec.trend_alpha))
        tau2[g] = rec.tau2
    return mu, alpha, tau2


def insample_shash_of(g, batch_id):
    rec = engine.genes[g]
    return rec.cv_shash_xi, rec.cv_shash_eta, rec.cv_shash_eps, rec.cv_shash_delta, bool(rec.cv_shash_ok)


results_insample = run_sweep(insample_mu_alpha_fn, insample_shash_of, lambda b: _insample_universe, "In-sample", "sweep_insample_v3.csv")

results = pd.concat([results_lobo, results_insample], ignore_index=True)

## Final comparison: LOBO vs In-sample x Stouffer vs MWU vs Wolfers

Precision alone is hard to read with only 250/~17,800 true positives (prevalence ~1.4%) -- also reporting **enrichment** (Precision / prevalence: how much better than randomly picking genes) and the gap to the ~95% Precision a correctly-calibrated FDR=5% test would give.

In [5]:
PREVALENCE = N_PERTURB_GENES / results["n_universe"].mean()


def summarize(df, label):
    g = df.groupby("log2fc")[["auc_stouffer", "sens_stouffer", "prec_stouffer",
                               "auc_mwu", "sens_mwu", "prec_mwu",
                               "auc_wolfers", "sens_wolfers", "prec_wolfers"]].mean().round(3)
    for m in ["stouffer", "mwu", "wolfers"]:
        g[f"enrich_{m}"] = (g[f"prec_{m}"] / PREVALENCE).round(1)
    g.columns = pd.MultiIndex.from_tuples([(label, c) for c in g.columns])
    return g


comparison = pd.concat([summarize(results_lobo, "LOBO"), summarize(results_insample, "In-sample")], axis=1)
print(f"prevalence = {PREVALENCE:.4f} ({N_PERTURB_GENES}/{results['n_universe'].mean():.0f}); "
      f"a well-calibrated FDR=5% test should give ~95% Precision regardless of prevalence")
comparison

prevalence = 0.0141 (250/17781); a well-calibrated FDR=5% test should give ~95% Precision regardless of prevalence


LOBO                                                        \
       auc_stouffer sens_stouffer prec_stouffer auc_mwu sens_mwu prec_mwu   
log2fc                                                                      
1.0           0.766         0.681         0.036   0.851    0.541    0.114   
2.0           0.876         0.835         0.044   0.920    0.737    0.144   
3.0           0.920         0.900         0.047   0.934    0.790    0.152   
4.0           0.951         0.945         0.049   0.957    0.849    0.161   
5.0           0.974         0.975         0.051   0.968    0.888    0.167   

                                                              ...  \
       auc_wolfers sens_wolfers prec_wolfers enrich_stouffer  ...   
log2fc                                                        ...   
1.0          0.907        0.661        0.245             2.6  ...   
2.0          0.965        0.885        0.248             3.1  ...   
3.0          0.985        0.944        0.260             3.3  ...   
4.0          0.993        0.974        0.266             3.5  ...   
5.0          0.998        0.990        0.269             3.6  ...   

           In-sample                                                     \
       prec_stouffer auc_mwu sens_mwu prec_mwu auc_wolfers sens_wolfers   
log2fc                                                                    
1.0            0.042   0.799    0.543    0.202       0.908        0.720   
2.0            0.052   0.894    0.737    0.249       0.974        0.904   
3.0            0.056   0.931    0.822    0.268       0.991        0.957   
4.0            0.059   0.949    0.860    0.274       0.996        0.981   
5.0            0.060   0.963    0.894    0.280       0.998        0.991   

                                                               
       prec_wolfers enrich_stouffer enrich_mwu enrich_wolfers  
log2fc                                                         
1.0           0.272             3.0       14.4           19.3  
2.0           0.321             3.7       17.7           22.8  
3.0           0.335             4.0       19.1           23.8  
4.0           0.340             4.2       19.5           24.2  
5.0           0.342             4.3       19.9           24.3  

[5 rows x 24 columns]